# LESSON 5.6: Reconstruction Filters
## Image Reconstruction from Projections

In this lesson:
- Why the pure ramp filter is problematic
- Windowed reconstruction filters (Ram-Lak, Shepp-Logan, Cosine, Hamming, Hann)
- Comparing filter effects on reconstruction quality
- Noise amplification and the noise-resolution trade-off
- Choosing the right filter for clinical applications

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon, iradon
from skimage.data import shepp_logan_phantom
from skimage.transform import rescale

## 1. The Problem with the Pure Ramp Filter

The ramp filter $H(\omega) = |\omega|$ is theoretically perfect, but in practice:

- It **amplifies high frequencies** linearly: the higher the frequency, the more gain
- **Noise** is typically concentrated at high frequencies
- Result: the ramp filter **amplifies noise** significantly

### Solution: Windowed Ramp Filters

Multiply the ramp filter by a **window function** $W(\omega)$ that attenuates high frequencies:

$$H_{\text{windowed}}(\omega) = |\omega| \cdot W(\omega)$$

This trades some **resolution** for better **noise suppression**.

In [ ]:
# Demonstrate noise amplification by ramp filter
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta = np.linspace(0., 180., 180, endpoint=False)
sinogram_clean = radon(phantom, theta=theta, circle=True)

# Add noise to sinogram
np.random.seed(42)
noise_std = 2.0
sinogram_noisy = sinogram_clean + np.random.normal(0, noise_std, sinogram_clean.shape)

# Reconstruct with ramp filter (no windowing)
recon_clean = iradon(sinogram_clean, theta=theta, filter_name='ramp', circle=True)
recon_noisy_ramp = iradon(sinogram_noisy, theta=theta, filter_name='ramp', circle=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(recon_clean, cmap='gray')
axes[0].set_title('FBP (clean data)', fontsize=12)

axes[1].imshow(recon_noisy_ramp, cmap='gray')
axes[1].set_title(f'FBP with ramp (noisy, σ={noise_std})', fontsize=12)

# Difference to show amplified noise
noise_amplified = recon_noisy_ramp - recon_clean
im = axes[2].imshow(noise_amplified, cmap='seismic', vmin=-0.5, vmax=0.5)
axes[2].set_title('Amplified Noise in Reconstruction', fontsize=12)
plt.colorbar(im, ax=axes[2])

plt.suptitle('Problem: Ramp Filter Amplifies Noise', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Noise std in sinogram: {noise_std}")
print(f"Noise std in reconstruction: {noise_amplified.std():.4f}")
print("The ramp filter has AMPLIFIED the noise significantly!")

## 2. Common Reconstruction Filters

All reconstruction filters have the form $H(\omega) = |\omega| \cdot W(\omega)$ where $W(\omega)$ is a window:

| Filter | Window $W(\omega)$ | Properties |
|--------|-------------------|------------|
| **Ram-Lak** (Ramp) | $W = 1$ | Maximum resolution, maximum noise |
| **Shepp-Logan** | $W = \text{sinc}(\omega / 2\omega_{\max})$ | Slight smoothing, good for medical imaging |
| **Cosine** | $W = \cos(\pi\omega / 2\omega_{\max})$ | Moderate smoothing |
| **Hamming** | $W = 0.54 + 0.46\cos(\pi\omega / \omega_{\max})$ | Strong smoothing, good noise suppression |
| **Hann** | $W = 0.5 + 0.5\cos(\pi\omega / \omega_{\max})$ | Similar to Hamming |

where $\omega_{\max}$ is the maximum (Nyquist) frequency.

In [ ]:
# Plot all reconstruction filters
n = 512
omega = np.linspace(-0.5, 0.5, n)
omega_max = 0.5

# Define filters
ramp = np.abs(omega)

# Shepp-Logan: |omega| * sinc(omega / (2*omega_max))
shepp_logan = np.abs(omega) * np.sinc(omega / (2 * omega_max))

# Cosine: |omega| * cos(pi*omega / (2*omega_max))
cosine = np.abs(omega) * np.cos(np.pi * omega / (2 * omega_max))

# Hamming: |omega| * (0.54 + 0.46*cos(pi*omega / omega_max))
hamming = np.abs(omega) * (0.54 + 0.46 * np.cos(np.pi * omega / omega_max))

# Hann: |omega| * (0.5 + 0.5*cos(pi*omega / omega_max))
hann = np.abs(omega) * (0.5 + 0.5 * np.cos(np.pi * omega / omega_max))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full filter H(omega)
axes[0].plot(omega, ramp, 'b-', linewidth=2, label='Ram-Lak (Ramp)')
axes[0].plot(omega, shepp_logan, 'r-', linewidth=2, label='Shepp-Logan')
axes[0].plot(omega, cosine, 'g-', linewidth=2, label='Cosine')
axes[0].plot(omega, hamming, 'm-', linewidth=2, label='Hamming')
axes[0].plot(omega, hann, 'c-', linewidth=2, label='Hann')
axes[0].set_title('Reconstruction Filters $H(\\omega) = |\\omega| \\cdot W(\\omega)$', fontsize=12)
axes[0].set_xlabel('Frequency $\\omega$')
axes[0].set_ylabel('$H(\\omega)$')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Window functions only
axes[1].plot(omega, np.ones_like(omega), 'b-', linewidth=2, label='Ram-Lak (W=1)')
axes[1].plot(omega, np.sinc(omega / (2 * omega_max)), 'r-', linewidth=2, label='Shepp-Logan')
axes[1].plot(omega, np.cos(np.pi * omega / (2 * omega_max)), 'g-', linewidth=2, label='Cosine')
axes[1].plot(omega, 0.54 + 0.46 * np.cos(np.pi * omega / omega_max), 'm-', linewidth=2, label='Hamming')
axes[1].plot(omega, 0.5 + 0.5 * np.cos(np.pi * omega / omega_max), 'c-', linewidth=2, label='Hann')
axes[1].set_title('Window Functions $W(\\omega)$', fontsize=12)
axes[1].set_xlabel('Frequency $\\omega$')
axes[1].set_ylabel('$W(\\omega)$')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Reconstruction Filters for FBP', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Ram-Lak: No windowing — maximum resolution but maximum noise")
print("Shepp-Logan: Slight roll-off — good compromise for medical imaging")
print("Cosine: Moderate roll-off — smoother reconstruction")
print("Hamming/Hann: Strong roll-off — smoothest, best noise suppression")

## 3. Comparing Filters on Clean Data

Let's see how each filter affects the reconstruction of noise-free data.

In [ ]:
# Compare filters on clean data
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta = np.linspace(0., 180., 180, endpoint=False)
sinogram = radon(phantom, theta=theta, circle=True)

filter_names = ['ramp', 'shepp-logan', 'cosine', 'hamming', 'hann']
filter_labels = ['Ram-Lak (Ramp)', 'Shepp-Logan', 'Cosine', 'Hamming', 'Hann']

fig, axes = plt.subplots(2, len(filter_names), figsize=(20, 8))

min_dim = None
for i, (fname, flabel) in enumerate(zip(filter_names, filter_labels)):
    recon = iradon(sinogram, theta=theta, filter_name=fname, circle=True)
    
    if min_dim is None:
        min_dim = min(phantom.shape[0], recon.shape[0])
    
    recon_crop = recon[:min_dim, :min_dim]
    phantom_crop = phantom[:min_dim, :min_dim]
    error = phantom_crop - recon_crop
    rmse = np.sqrt(np.mean(error**2))
    
    axes[0, i].imshow(recon_crop, cmap='gray')
    axes[0, i].set_title(f'{flabel}', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(np.abs(error), cmap='hot', vmin=0, vmax=0.1)
    axes[1, i].set_title(f'RMSE = {rmse:.4f}', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Reconstruction', fontsize=11)
axes[1, 0].set_ylabel('|Error|', fontsize=11)

plt.suptitle('Filter Comparison on Clean Data (180 projections)',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("On clean data, Ram-Lak gives the best reconstruction (lowest error).")
print("Windowed filters introduce slight blurring, increasing the error.")
print("But the story changes with noisy data...")

## 4. Comparing Filters on Noisy Data

The real advantage of windowed filters becomes apparent with noisy projections.

In [ ]:
# Compare filters on noisy data
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta = np.linspace(0., 180., 180, endpoint=False)
sinogram_clean = radon(phantom, theta=theta, circle=True)

np.random.seed(42)
noise_std = 3.0
sinogram_noisy = sinogram_clean + np.random.normal(0, noise_std, sinogram_clean.shape)

filter_names = ['ramp', 'shepp-logan', 'cosine', 'hamming', 'hann']
filter_labels = ['Ram-Lak', 'Shepp-Logan', 'Cosine', 'Hamming', 'Hann']

fig, axes = plt.subplots(2, len(filter_names), figsize=(20, 8))

rmse_values = []
min_dim = None
for i, (fname, flabel) in enumerate(zip(filter_names, filter_labels)):
    recon = iradon(sinogram_noisy, theta=theta, filter_name=fname, circle=True)
    
    if min_dim is None:
        min_dim = min(phantom.shape[0], recon.shape[0])
    
    recon_crop = recon[:min_dim, :min_dim]
    phantom_crop = phantom[:min_dim, :min_dim]
    error = phantom_crop - recon_crop
    rmse = np.sqrt(np.mean(error**2))
    rmse_values.append(rmse)
    
    axes[0, i].imshow(recon_crop, cmap='gray')
    axes[0, i].set_title(f'{flabel}', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(np.abs(error), cmap='hot', vmin=0, vmax=0.5)
    axes[1, i].set_title(f'RMSE = {rmse:.4f}', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Reconstruction', fontsize=11)
axes[1, 0].set_ylabel('|Error|', fontsize=11)

plt.suptitle(f'Filter Comparison on Noisy Data (σ = {noise_std})',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nRMSE values:")
for fname, rmse in zip(filter_labels, rmse_values):
    print(f"  {fname:15s}: {rmse:.4f}")
print(f"\nBest filter for noisy data: {filter_labels[np.argmin(rmse_values)]}")
print("With noise, windowed filters (Hamming/Hann) outperform the pure ramp!")

## 5. The Noise-Resolution Trade-Off

There is a fundamental trade-off in CT reconstruction:

| More high-frequency gain | Less high-frequency gain |
|---|---|
| Better spatial resolution | Worse spatial resolution |
| More noise | Less noise |
| Ram-Lak filter | Hamming filter |
| Sharp edges, noisy regions | Smooth edges, clean regions |

The choice depends on the clinical application.

In [ ]:
# Noise-resolution trade-off analysis
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta = np.linspace(0., 180., 180, endpoint=False)
sinogram_clean = radon(phantom, theta=theta, circle=True)

# Test over multiple noise levels
noise_levels = np.linspace(0, 5, 20)
filter_names = ['ramp', 'shepp-logan', 'cosine', 'hamming']
filter_labels = ['Ram-Lak', 'Shepp-Logan', 'Cosine', 'Hamming']
colors = ['blue', 'red', 'green', 'purple']

rmse_results = {fname: [] for fname in filter_names}

np.random.seed(42)
for noise_std in noise_levels:
    if noise_std == 0:
        sinogram = sinogram_clean.copy()
    else:
        sinogram = sinogram_clean + np.random.normal(0, noise_std, sinogram_clean.shape)
    
    for fname in filter_names:
        recon = iradon(sinogram, theta=theta, filter_name=fname, circle=True)
        min_dim = min(phantom.shape[0], recon.shape[0])
        error = phantom[:min_dim, :min_dim] - recon[:min_dim, :min_dim]
        rmse = np.sqrt(np.mean(error**2))
        rmse_results[fname].append(rmse)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for fname, flabel, color in zip(filter_names, filter_labels, colors):
    axes[0].plot(noise_levels, rmse_results[fname], color=color,
               linewidth=2, label=flabel, marker='o', markersize=3)

axes[0].set_title('RMSE vs Noise Level', fontsize=12)
axes[0].set_xlabel('Noise σ')
axes[0].set_ylabel('RMSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Show the crossover point
axes[1].plot(noise_levels, rmse_results['ramp'], 'b-', linewidth=2,
            label='Ram-Lak', marker='o', markersize=3)
axes[1].plot(noise_levels, rmse_results['hamming'], 'm-', linewidth=2,
            label='Hamming', marker='s', markersize=3)
axes[1].fill_between(noise_levels,
                    rmse_results['ramp'], rmse_results['hamming'],
                    where=[r < h for r, h in zip(rmse_results['ramp'], rmse_results['hamming'])],
                    alpha=0.2, color='blue', label='Ram-Lak better')
axes[1].fill_between(noise_levels,
                    rmse_results['ramp'], rmse_results['hamming'],
                    where=[r >= h for r, h in zip(rmse_results['ramp'], rmse_results['hamming'])],
                    alpha=0.2, color='purple', label='Hamming better')
axes[1].set_title('Crossover: Ram-Lak vs Hamming', fontsize=12)
axes[1].set_xlabel('Noise σ')
axes[1].set_ylabel('RMSE')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('The Noise-Resolution Trade-Off', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("At low noise: Ram-Lak is best (preserves resolution without penalty).")
print("At high noise: Hamming/Hann are best (smoothing outweighs resolution loss).")
print("Shepp-Logan provides a good compromise for moderate noise levels.")

## 6. Line Profile Analysis

Let's examine how each filter affects edge sharpness and noise in line profiles.

In [ ]:
# Line profile comparison
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta = np.linspace(0., 180., 180, endpoint=False)
sinogram_clean = radon(phantom, theta=theta, circle=True)

np.random.seed(42)
sinogram_noisy = sinogram_clean + np.random.normal(0, 3.0, sinogram_clean.shape)

filter_names = ['ramp', 'shepp-logan', 'cosine', 'hamming']
filter_labels = ['Ram-Lak', 'Shepp-Logan', 'Cosine', 'Hamming']
colors = ['blue', 'red', 'green', 'purple']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

min_dim = None
for fname, flabel, color in zip(filter_names, filter_labels, colors):
    recon = iradon(sinogram_noisy, theta=theta, filter_name=fname, circle=True)
    if min_dim is None:
        min_dim = min(phantom.shape[0], recon.shape[0])
    
    center = min_dim // 2
    axes[0].plot(recon[center, :min_dim], color=color, linewidth=1.5, label=flabel, alpha=0.8)
    axes[1].plot(recon[:min_dim, center], color=color, linewidth=1.5, label=flabel, alpha=0.8)

# Add ground truth
center = min_dim // 2
axes[0].plot(phantom[center, :min_dim], 'k-', linewidth=2, label='Ground Truth')
axes[0].set_title('Horizontal Profile (center row)', fontsize=12)
axes[0].set_xlabel('Pixel position')
axes[0].set_ylabel('Value')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(phantom[:min_dim, center], 'k-', linewidth=2, label='Ground Truth')
axes[1].set_title('Vertical Profile (center column)', fontsize=12)
axes[1].set_xlabel('Pixel position')
axes[1].set_ylabel('Value')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Line Profiles: Edge Sharpness vs Noise (σ = 3.0)',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Ram-Lak (blue): Sharpest edges but most noisy fluctuations.")
print("Hamming (purple): Smoothest but edges are slightly blurred.")
print("Shepp-Logan (red): Good balance between sharpness and smoothness.")

## 7. Clinical Filter Selection Guide

| Application | Recommended Filter | Reason |
|------------|-------------------|--------|
| **High-contrast imaging** (bones, lung) | Ram-Lak or Shepp-Logan | Resolution matters, contrast is high |
| **Soft tissue imaging** (brain, abdomen) | Hamming or Hann | Noise suppression critical for low-contrast |
| **General purpose** | Shepp-Logan or Cosine | Good compromise |
| **Low-dose CT** | Hamming | Maximum noise suppression needed |
| **Research / phantom studies** | Ram-Lak | Clean data, maximize resolution |

In [ ]:
# Simulated clinical scenarios
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)

theta = np.linspace(0., 180., 180, endpoint=False)
sinogram_clean = radon(phantom, theta=theta, circle=True)

# Different noise levels simulating different dose levels
np.random.seed(42)
scenarios = [
    ('High Dose (σ=0.5)', 0.5, 'ramp'),
    ('Standard Dose (σ=2.0)', 2.0, 'shepp-logan'),
    ('Low Dose (σ=5.0)', 5.0, 'hamming'),
]

fig, axes = plt.subplots(2, len(scenarios), figsize=(15, 9))

for i, (scenario_name, noise_std, rec_filter) in enumerate(scenarios):
    sinogram_noisy = sinogram_clean + np.random.normal(0, noise_std, sinogram_clean.shape)
    
    # Reconstruct with recommended filter
    recon = iradon(sinogram_noisy, theta=theta, filter_name=rec_filter, circle=True)
    
    # Also reconstruct with ramp for comparison
    recon_ramp = iradon(sinogram_noisy, theta=theta, filter_name='ramp', circle=True)
    
    axes[0, i].imshow(recon_ramp, cmap='gray')
    axes[0, i].set_title(f'{scenario_name}\nRam-Lak filter', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(recon, cmap='gray')
    axes[1, i].set_title(f'{scenario_name}\nRecommended: {rec_filter}', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Ram-Lak\n(always)', fontsize=11)
axes[1, 0].set_ylabel('Recommended\nfilter', fontsize=11)

plt.suptitle('Clinical Scenarios: Choosing the Right Filter',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("High dose → Low noise → Ram-Lak gives excellent results.")
print("Standard dose → Moderate noise → Shepp-Logan balances resolution and noise.")
print("Low dose → High noise → Hamming essential for acceptable image quality.")

## Summary

What we learned:
1. The pure **ramp (Ram-Lak) filter** amplifies noise proportionally to frequency
2. **Windowed filters** multiply the ramp by a window function to attenuate high frequencies
3. Common filters: **Ram-Lak** (sharpest), **Shepp-Logan** (slight smoothing), **Cosine**, **Hamming** (smoothest)
4. There is a fundamental **noise-resolution trade-off**: more smoothing = less noise but less resolution
5. At **low noise**, Ram-Lak is optimal; at **high noise**, Hamming/Hann outperform
6. The **Shepp-Logan filter** is often the best general-purpose choice for medical imaging
7. Filter selection in clinical CT depends on the **imaging task** and **radiation dose**